# Resume Roundabout Training

Continue an existing DQN or PPO run from its latest checkpoint. The setup below follows `main_training.ipynb`: clone the training branch, install SUMO and project dependencies, mount Google Drive, and link `outputs/runs` to the same `MyDrive/traffic_drl_runs` folder. Then enter the run directory name once in the next cell. That value selects the checkpoint and keeps resumed outputs under the same run.

Outside Colab, use a compatible local SUMO installation and the checked-out repository; the notebook uses the local `outputs/runs` folder.

## 1. Repository and SUMO setup

In Colab, clone the same `main-training` branch as the original notebook. Locally, find the existing project checkout instead.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

project_root = Path.cwd()
if "google.colab" in sys.modules:
    repo_dir = Path("/content/DRL-Traffic-Signal-Control")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--branch", "main-training", "https://github.com/TCL03-HCMUT/DRL-Traffic-Signal-Control.git", str(repo_dir)],
            check=True,
        )
    project_root = repo_dir
    os.chdir(project_root)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.home() / "DRL-Traffic-Signal-Control"]
    for candidate in candidates:
        if (candidate / "src" / "traffic_drl").exists():
            project_root = candidate
            break
    os.chdir(project_root)

if not (project_root / "src" / "traffic_drl").exists():
    raise FileNotFoundError(f"Project source not found under {project_root}")
print(f"Project root: {project_root}")

In [ ]:
# Install SUMO and project dependencies in Colab, matching main_training.ipynb.
if "google.colab" in sys.modules:
    subprocess.run(["sudo", "add-apt-repository", "-y", "ppa:sumo/stable"], check=True)
    subprocess.run(["sudo", "apt-get", "update", "-y"], check=True)
    subprocess.run(["sudo", "apt-get", "install", "-y", "sumo", "sumo-tools", "sumo-doc"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "."], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "gym"], check=True)
    os.environ["SUMO_HOME"] = "/usr/share/sumo"
    os.environ["LIBSUMO_AS_TRACI"] = "1"
else:
    print("Using the local Python environment and installed SUMO runtime.")

In [ ]:
import torch
import traci
import sumo_rl
import gymnasium as gym
import stable_baselines3

if "google.colab" in sys.modules:
    os.environ["SUMO_HOME"] = "/usr/share/sumo"
elif "SUMO_HOME" not in os.environ:
    for candidate in [Path("/usr/share/sumo"), Path("C:/Program Files (x86)/Eclipse/Sumo"), project_root / "sumo"]:
        if candidate.exists():
            os.environ["SUMO_HOME"] = str(candidate)
            break

print(f"SUMO_HOME: {os.environ.get('SUMO_HOME')}")
print(f"PyTorch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"Gymnasium: {gym.__version__}")
print(f"Stable-Baselines3: {stable_baselines3.__version__}")
print(f"TraCI: {traci.__version__}")
print(f"SUMO-RL: {sumo_rl.__file__}")

In [ ]:
import subprocess

subprocess.run(["sumo", "--version"], check=True)

In [ ]:
# Regenerate route files from the checked-out scenario manifest for a fresh Colab runtime.
import csv
import subprocess

csv_path = Path("scenarios/json_definitions/selected_scenario.csv")
factory_script = Path("scenarios/generators/scenario_factory.py")
with csv_path.open("r", encoding="utf-8") as scenario_file:
    scenario_ids = [row["scenario_id"] for row in csv.DictReader(scenario_file) if row.get("scenario_id")]

if not scenario_ids:
    raise ValueError(f"No scenarios found in {csv_path}")
for scenario_id in scenario_ids:
    subprocess.run(
        [sys.executable, str(factory_script), "--scenario", scenario_id],
        check=True,
        text=True,
    )
print(f"Generated {len(scenario_ids)} scenarios.")

## 2. Google Drive run storage

Mount the same Drive folder used by `main_training.ipynb` so the selected run's checkpoints are available.

In [ ]:
# Mount persistent storage and point outputs/runs at the same Drive folder as the main notebook.
from pathlib import Path
import sys

RUNS_DIR = Path("outputs/runs")
if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    gdrive_target = Path("/content/drive/MyDrive/traffic_drl_runs")
    gdrive_target.mkdir(parents=True, exist_ok=True)
    RUNS_DIR.parent.mkdir(parents=True, exist_ok=True)

    if RUNS_DIR.is_symlink():
        if RUNS_DIR.resolve() != gdrive_target.resolve():
            RUNS_DIR.unlink()
    elif RUNS_DIR.exists():
        if any(RUNS_DIR.iterdir()):
            local_backup = RUNS_DIR.with_name("runs_local_backup")
            if local_backup.exists():
                raise FileExistsError(
                    f"Cannot preserve local runs because backup already exists: {local_backup}"
                )
            RUNS_DIR.rename(local_backup)
            print(f"Preserved existing local runs at {local_backup}")
        else:
            RUNS_DIR.rmdir()

    if not RUNS_DIR.exists():
        RUNS_DIR.symlink_to(gdrive_target, target_is_directory=True)
    print(f"Run storage: {RUNS_DIR.resolve()}")
else:
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Local run storage: {RUNS_DIR.resolve()}")

In [ ]:
from pathlib import Path
import json
import os

# Enter the run directory name once. It is reused for loading and saving.
RUN_ID = "<run_id>"

# Additional environment steps to train after the checkpoint.
TRAIN_TIMESTEPS = 100_000

RUN_DIR = RUNS_DIR / RUN_ID
CHECKPOINT_DIR = RUN_DIR / "checkpoints" / "last"
RESUME_INFO_PATH = CHECKPOINT_DIR / "resume_info.json"
MODEL_PATH = CHECKPOINT_DIR / "model.zip"

if not RUN_ID.strip() or RUN_ID.startswith("<"):
    raise ValueError("Set RUN_ID to the existing run directory name before continuing.")
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint model not found: {MODEL_PATH}")
if not RESUME_INFO_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint metadata not found: {RESUME_INFO_PATH}")

resume_info = json.loads(RESUME_INFO_PATH.read_text(encoding="utf-8"))
MODEL_CLASS = resume_info.get("model_class")
ALGORITHM_BY_CLASS = {"DQN": "DQN", "PPO": "PPO"}
if MODEL_CLASS not in ALGORITHM_BY_CLASS:
    raise ValueError(f"Unsupported checkpoint model_class: {MODEL_CLASS!r}")
SELECTED_ALGO = ALGORITHM_BY_CLASS[MODEL_CLASS]
print(f"Run: {RUN_ID}")
print(f"Algorithm: {SELECTED_ALGO}")
print(f"Checkpoint step: {resume_info.get('num_timesteps', 'unknown')}")
print(f"Additional training steps: {TRAIN_TIMESTEPS}")

In [ ]:
from traffic_drl.config import load_env_config, load_train_config
from traffic_drl.environment.make_env import build_reward_fn, make_vectorized_environment
from traffic_drl.environment.scenario_factory import ScenarioManifest
from traffic_drl.train.scenario_sampler import ScenarioSampler

CONFIG_PATH_BY_ALGO = {
    "DQN": "configs/train/dqn_roundabout.yaml",
    "PPO": "configs/train/ppo_roundabout.yaml",
}
active_config = load_train_config(CONFIG_PATH_BY_ALGO[SELECTED_ALGO])
env_config = load_env_config(active_config.environment.env_config_path)

manifest = ScenarioManifest.from_csv(active_config.environment.manifest_path)
sampler = ScenarioSampler(
    manifest=manifest,
    allowed_split=active_config.experiment.split_allowed,
    seed=active_config.experiment.seed,
    difficulty_unlock_timesteps=(0, 300000, 600000),
)
initial_record = sampler.sample()
reward_fn = build_reward_fn(active_config.reward)

# Match the main notebook's worker count; lower this if the runtime is constrained.
N_ENVS = max(1, os.cpu_count() or 1)

train_env = make_vectorized_environment(
    config=env_config,
    route_file=initial_record.route_file,
    run_id=RUN_ID,
    base_dir=RUNS_DIR,
    seed=active_config.experiment.seed,
    reward_fn=reward_fn,
    wrap=True,
    scenario_sampler=sampler,
    custom_observation=True,
    training=True,
    # load_*_checkpoint restores the saved VecNormalize wrapper around this raw env.
    norm_obs=False,
    norm_reward=False,
    approach_ids=env_config.observation_features.approach_ids,
    ring_segment_ids=env_config.observation_features.ring_segment_ids,
    vehicle_classes=env_config.observation_features.vehicle_classes,
    n_envs=N_ENVS,
    start_method="spawn",
)

print(f"Training scenarios restricted to split: {active_config.experiment.split_allowed}")
print(f"Initial route: {initial_record.scenario_id} -> {initial_record.route_file}")
print(f"Vector environment workers: {N_ENVS}")
print(f"Observation space: {train_env.observation_space}")
print(f"Action space: {train_env.action_space}")

In [ ]:
from stable_baselines3.common.callbacks import CallbackList
from traffic_drl.train.callbacks import CurriculumCallback, RobustCheckpointCallback, TrafficMetricsCallback
from traffic_drl.train.train_dqn import load_dqn_checkpoint, train_dqn
from traffic_drl.train.train_ppo import load_ppo_checkpoint, train_ppo

checkpoint_callback = RobustCheckpointCallback(
    save_freq=active_config.training_control.save_freq,
    save_dir=RUN_DIR / "checkpoints",
    save_replay_buffer=(SELECTED_ALGO == "DQN"),
    save_vec_normalize=True,
    verbose=1,
)
curriculum_callback = CurriculumCallback(scenario_sampler=sampler)
callbacks = CallbackList([checkpoint_callback, TrafficMetricsCallback(), curriculum_callback])

try:
    if SELECTED_ALGO == "DQN":
        model = load_dqn_checkpoint(CHECKPOINT_DIR, env=train_env, training=True)
        trained_model = train_dqn(
            model,
            total_timesteps=TRAIN_TIMESTEPS,
            callback=callbacks,
            reset_num_timesteps=False,
        )
    else:
        model = load_ppo_checkpoint(CHECKPOINT_DIR, env=train_env, training=True)
        trained_model = train_ppo(
            model,
            total_timesteps=TRAIN_TIMESTEPS,
            callback=callbacks,
            reset_num_timesteps=False,
        )
finally:
    train_env.close()

print("Resume training complete.")
print(f"Updated checkpoint directory: {CHECKPOINT_DIR}")
print(f"Total model timesteps: {trained_model.num_timesteps}")

The `last/` checkpoint is updated periodically and once more when training ends. The run ID stays unchanged, so results, trip information, logs, and checkpoints continue under the same run directory. `TRAIN_TIMESTEPS` is the number of additional steps, not the target cumulative step count.